# 04. Direct Preference Optimization on a small model

A companion to the [post-training primer](https://www.norange.ai/post-training/2026/08/24/post-training-primer.html).

Train **Qwen2.5-0.5B** with LoRA on **256 synthetic preference pairs**. We reuse the request-to-JSON task from notebook 03, but replace demonstration likelihood training with the **DPO loss**. The chosen response follows the request; the rejected response has a wrong field or extra text. This is a transparent, rule-labeled preference dataset, not human preference data.

Use **Runtime → Change runtime type → T4 GPU** and run all cells. No earlier notebook or saved adapter is required. Short sequences and 96 optimizer updates keep the experiment small; actual training time is printed. We do not run SFT first: the reference and starting policy are the same pretrained base checkpoint, so the before/after change isolates this DPO run.

We measure actual greedy generations on 64 held-out requests before training and after each epoch. Results include content accuracy, standalone JSON, exact task success, paired improvements/regressions, and raw outputs. DPO loss and preference margins are separate diagnostics; a better margin does not by itself imply better generated answers.


In [ ]:
%pip -q install "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" pandas matplotlib tqdm


In [ ]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, set_seed
from peft import LoraConfig, TaskType, get_peft_model

SEED = 7
MODEL_ID = "Qwen/Qwen2.5-0.5B"
TRAIN_SIZE = 256
TEST_SIZE = 64
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 1e-4
BETA = 0.1
ACCUMULATION_STEPS = 2
MAX_LENGTH = 192
MAX_NEW_TOKENS = 64

set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU in Runtime → Change runtime type, then rerun.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("Training updates:", math.ceil(TRAIN_SIZE / (BATCH_SIZE * ACCUMULATION_STEPS)) * EPOCHS)


## Build fixed training and held-out preference pairs

The requests use the same deterministic split as notebook 03: no (item, color, quantity) combination occurs in both training and evaluation. Half of evaluation requests also use unseen sentence templates. Individual field values are familiar.

Three quarters of the pairs compare two JSON objects of similar length, with exactly one field corrupted. The remaining quarter compares a correct standalone object with that same object surrounded by unwanted prose. Thus shorter output alone cannot satisfy the task. Labels are generated from known request fields; no reward model or judge is trained.


In [ ]:
COLORS = ["red", "blue", "green", "black", "white", "yellow", "pink", "purple"]
ITEMS = ["mug", "notebook", "bottle", "lamp", "plate", "scarf", "wallet", "backpack"]
TRAIN_TEMPLATES = [
    "Please order {quantity} units of the {color} {item}.",
    "I want a {color} {item}. Quantity: {quantity}.",
    "Add to my order: {item}, color {color}, quantity {quantity}.",
    "Can you send {quantity} of the {item} in {color}?",
]
UNSEEN_TEMPLATES = [
    "The {item} should be {color}; send me {quantity} in total.",
    "For this purchase, choose {color} for the {item} and set the count to {quantity}.",
]
INSTRUCTION = (
    'Extract the order from the request. Return exactly one JSON object with keys '
    '"item", "color", and "quantity". Use lowercase strings and an integer quantity. '
    'Do not include explanations or markdown.\n'
)

rng = random.Random(SEED)
combinations = list(itertools.product(ITEMS, COLORS, range(1, 10)))
rng.shuffle(combinations)

def make_record(values, template, group):
    item, color, quantity = values
    target = dict(item=item, color=color, quantity=quantity)
    return dict(request=template.format(**target), target=target, group=group)

train_records = [
    make_record(values, TRAIN_TEMPLATES[i % len(TRAIN_TEMPLATES)], "train")
    for i, values in enumerate(combinations[:TRAIN_SIZE])
]
test_records = []
for i, values in enumerate(combinations[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]):
    familiar = i < TEST_SIZE // 2
    templates = TRAIN_TEMPLATES if familiar else UNSEEN_TEMPLATES
    test_records.append(make_record(
        values, templates[i % len(templates)],
        "New combinations" if familiar else "New wording",
    ))


def target_text(record):
    return json.dumps(record["target"], ensure_ascii=False)

def prompt_for(record, examples=()):
    demonstrations = "".join(
        f'Request: {example["request"]}\nJSON: {target_text(example)}\n\n'
        for example in examples
    )
    return INSTRUCTION + demonstrations + f'Request: {record["request"]}\nJSON: '

def record_key(record):
    return tuple(record["target"][key] for key in ("item", "color", "quantity"))

assert len(train_records) == TRAIN_SIZE and len(test_records) == TEST_SIZE
assert not {record_key(r) for r in train_records} & {record_key(r) for r in test_records}
assert not {r["request"] for r in train_records} & {r["request"] for r in test_records}
for key in ("item", "color", "quantity"):
    assert {r["target"][key] for r in test_records} <= {r["target"][key] for r in train_records}
display(pd.DataFrame([
    {"request": r["request"], "target": target_text(r)} for r in train_records[:4]
]))


In [ ]:
def make_preference(record, kind):
    chosen = target_text(record)
    wrong = dict(record["target"])
    if kind == "wrong color":
        wrong["color"] = COLORS[(COLORS.index(wrong["color"]) + 1) % len(COLORS)]
    elif kind == "wrong item":
        wrong["item"] = ITEMS[(ITEMS.index(wrong["item"]) + 1) % len(ITEMS)]
    elif kind == "wrong quantity":
        wrong["quantity"] = wrong["quantity"] % 9 + 1
    rejected = ("Here is your order: " + chosen + " Thank you!" if kind == "extra text"
                else json.dumps(wrong))
    return {**record, "chosen": chosen, "rejected": rejected, "kind": kind}

def build_pairs(records, seed):
    kinds = ["wrong color", "wrong item", "wrong quantity", "extra text"]
    assignments = [kinds[i % 4] for i in range(len(records))]
    random.Random(seed).shuffle(assignments)
    return [make_preference(record, kind) for record, kind in zip(records, assignments, strict=True)]

train_pairs = build_pairs(train_records, SEED + 1)
test_pairs = build_pairs(test_records, SEED + 2)
assert all(p["chosen"] != p["rejected"] for p in train_pairs + test_pairs)
display(pd.DataFrame(train_pairs[:4])[["request", "chosen", "rejected", "kind"]])


## Load the policy and measure the starting point

We use the base model and the same instruction-only prompt in every evaluation. There are no demonstrations in evaluation prompts and no grammar-constrained decoder. All generations are greedy, with the same maximum output length before and after training.

**Standalone JSON** checks the entire output. **Correct content** extracts the first decodable JSON object even if surrounded by prose. **Exact task success** additionally requires exactly the requested fields and correct types, without extra text. Whitespace and key order do not affect the score.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
# Left padding is needed for batched decoder-only generation.
tokenizer.padding_side = "left"
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, attn_implementation="sdpa",
).to(DEVICE)
base_model.eval()

generation_config = GenerationConfig(
    do_sample=False,
    max_new_tokens=MAX_NEW_TOKENS,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=True,
)

@torch.inference_mode()
def generate_responses(model, records, examples=()):
    model.eval()
    outputs = []
    for start in tqdm(range(0, len(records), EVAL_BATCH_SIZE), desc="Generate"):
        batch = records[start:start + EVAL_BATCH_SIZE]
        prompts = [prompt_for(record, examples) for record in batch]
        inputs = tokenizer(prompts, padding=True, return_tensors="pt",
                           add_special_tokens=False).to(DEVICE)
        generated = model.generate(**inputs, generation_config=generation_config)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return outputs


In [ ]:
def score_response(text, target):
    try:
        strict = json.loads(text.strip())
    except (json.JSONDecodeError, ValueError):
        strict = None
    json_only = isinstance(strict, dict)
    # For the content diagnostic, inspect only the first decodable object.
    parsed = strict if json_only else None
    if parsed is None:
        for i, character in enumerate(text):
            if character != "{":
                continue
            try:
                candidate, _ = json.JSONDecoder().raw_decode(text[i:])
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict):
                parsed = candidate
                break
    parsed = parsed if isinstance(parsed, dict) else {}
    content = all(
        type(parsed.get(key)) is type(value) and parsed.get(key) == value
        for key, value in target.items()
    )
    return {
        "Standalone JSON": json_only,
        "Correct content": content,
        "Exact task success": json_only and content and set(parsed) == set(target),
    }

def evaluate(outputs, records, condition):
    return pd.DataFrame([
        {"condition": condition, "group": record["group"], "request": record["request"],
         "target": target_text(record), "output": output,
         **score_response(output, record["target"])}
        for record, output in zip(records, outputs, strict=True)
    ])

# Sanity checks ensure that prose and incorrect types are not counted as exact success.
toy = {"item": "mug", "color": "red", "quantity": 2}
assert score_response(json.dumps(toy), toy)["Exact task success"]
assert not score_response("Here: " + json.dumps(toy), toy)["Exact task success"]
assert score_response("Here: " + json.dumps(toy), toy)["Correct content"]
assert not score_response('{"item":"mug","color":"red","quantity":"2"}', toy)["Correct content"]
assert not score_response("No JSON here", toy)["Standalone JSON"]


for pair in train_pairs + test_pairs:
    assert score_response(pair["chosen"], pair["target"])["Exact task success"]
    assert not score_response(pair["rejected"], pair["target"])["Exact task success"]

before_outputs = generate_responses(base_model, test_records)
metrics = ["Standalone JSON", "Correct content", "Exact task success"]
before = evaluate(before_outputs, test_records, "Before DPO")
all_results = [before.assign(epoch=0)]
outputs_by_epoch = {0: before_outputs}
display(before.groupby("group")[metrics].mean().mul(100).round(1))


## Compute completion log probabilities

For each response we sum the conditional log probabilities of its tokens **and EOS**, excluding prompt and padding tokens:

$$\log\pi_\theta(o\mid q)=\sum_{t\in\mathrm{completion}}\log\pi_\theta(o_t\mid q,o_{<t}).$$

As in SFT, prefixes come from the recorded response. Here we need a separate sequence log probability for each chosen/rejected completion, rather than a single averaged language-model loss. We shift predictions and labels exactly once. We use **sums**, not token averages, as required by ordinary DPO.


In [ ]:
def encode_completion(record, completion):
    prefix = tokenizer.encode(prompt_for(record), add_special_tokens=False)
    answer = tokenizer.encode(completion, add_special_tokens=False) + [tokenizer.eos_token_id]
    assert len(prefix) + len(answer) <= MAX_LENGTH, "Increase MAX_LENGTH; never truncate labels."
    return {"input_ids": prefix + answer, "labels": [-100] * len(prefix) + answer}

def pack_pairs(pairs):
    # First all chosen responses, then all rejected responses.
    rows = [encode_completion(p, p[key]) for key in ("chosen", "rejected") for p in pairs]
    width = max(len(row["input_ids"]) for row in rows)
    ids = torch.full((len(rows), width), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    attention = torch.zeros_like(ids)
    for i, row in enumerate(rows):
        size = len(row["input_ids"])
        ids[i, :size] = torch.tensor(row["input_ids"])
        labels[i, :size] = torch.tensor(row["labels"])
        attention[i, :size] = 1
    return {"input_ids": ids.to(DEVICE), "attention_mask": attention.to(DEVICE),
            "labels": labels.to(DEVICE)}

def sequence_logps(policy, batch):
    logits = policy(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    use_cache=False).logits[:, :-1, :]
    labels = batch["labels"][:, 1:]
    mask = labels != -100
    # Select only completion positions before computing float32 cross-entropy.
    token_logps = -F.cross_entropy(logits[mask].float(), labels[mask], reduction="none")
    row_indices = mask.nonzero(as_tuple=True)[0]
    return torch.zeros(labels.shape[0], device=logits.device, dtype=torch.float32).scatter_add(
        0, row_indices, token_logps,
    )

def pair_logps(policy, pairs):
    values = sequence_logps(policy, pack_pairs(pairs))
    return values[:len(pairs)], values[len(pairs):]

# Check prompt masking, padding masking, and inclusion of EOS.
probe = pack_pairs(train_pairs[:2])
for row in range(probe["input_ids"].shape[0]):
    valid = probe["labels"][row] != -100
    assert valid.any() and not valid[0]
    assert probe["labels"][row, valid][-1].item() == tokenizer.eos_token_id
assert (probe["labels"][probe["attention_mask"] == 0] == -100).all()
# Check the sequence scorer against the model's own shifted token loss.
with torch.no_grad():
    expected_nll = base_model(**probe, use_cache=False).loss.float()
    token_count = (probe["labels"][:, 1:] != -100).sum()
    actual_nll = -sequence_logps(base_model, probe).sum() / token_count
assert torch.allclose(expected_nll, actual_nll, atol=0.02, rtol=0.01)
del probe


## Freeze the reference by caching its scores

Before any updates, compute the reference log probabilities for every fixed pair. These detached CPU tensors remain unchanged throughout training. This is equivalent to evaluating a frozen reference on those same responses, without keeping a second model on the GPU. Do not recompute them from the updated policy.

The held-out cache is used only for reporting diagnostic margins. Its examples never enter the training loop, and we use the final fixed epoch rather than selecting a checkpoint on held-out scores.


In [ ]:
@torch.inference_mode()
def cache_logps(policy, pairs):
    policy.eval()
    values = []
    for start in tqdm(range(0, len(pairs), BATCH_SIZE), desc="Reference log probabilities"):
        chosen, rejected = pair_logps(policy, pairs[start:start + BATCH_SIZE])
        values.append(torch.stack([chosen, rejected], dim=1).cpu())
    return torch.cat(values).clone()

reference_train = cache_logps(base_model, train_pairs)
reference_test = cache_logps(base_model, test_pairs)
reference_snapshot = reference_train.clone()
assert not reference_train.requires_grad and torch.isfinite(reference_train).all()


## The DPO loss, explicitly

For each pair:

$$m_\theta=\beta\left[(\log\pi_\theta(o^+\mid q)-\log\pi_{\mathrm{ref}}(o^+\mid q))-(\log\pi_\theta(o^-\mid q)-\log\pi_{\mathrm{ref}}(o^-\mid q))\right],$$
$$\ell_{\mathrm{DPO}}=-\log\sigma(m_\theta).$$

Initially policy and reference match, so the margin is zero and the loss is $\log 2$, regardless of which response has the larger raw likelihood. DPO increases the chosen-versus-rejected log-probability gap **relative to the frozen reference**. A positive margin alone does not imply that the chosen response has the highest absolute likelihood or will be generated.

There is no reward-model training, actor advantage, or SFT auxiliary loss here. LoRA supplies the trainable parameters; ordinary backpropagation optimizes the preference loss.


In [ ]:
def dpo_loss(chosen_logps, rejected_logps, reference_logps):
    margin = BETA * ((chosen_logps - reference_logps[:, 0]) -
                     (rejected_logps - reference_logps[:, 1]))
    return -F.logsigmoid(margin), margin

# Small differentiable checks of signs and initialization.
test_chosen = torch.tensor([-3.0], requires_grad=True)
test_rejected = torch.tensor([-4.0], requires_grad=True)
test_reference = torch.tensor([[-3.0, -4.0]])
losses, margins = dpo_loss(test_chosen, test_rejected, test_reference)
assert torch.allclose(losses, torch.tensor([math.log(2)]))
losses.mean().backward()
assert test_chosen.grad.item() < 0 and test_rejected.grad.item() > 0
assert dpo_loss(test_chosen.detach() + 1, test_rejected.detach(), test_reference)[0].item() < math.log(2)

set_seed(SEED)
model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
))
model.print_trainable_parameters()
model.eval()
with torch.no_grad():
    cp, rp = pair_logps(model, train_pairs[:BATCH_SIZE])
    initial_loss, initial_margin = dpo_loss(cp, rp, reference_train[:BATCH_SIZE].to(DEVICE))
assert initial_margin.abs().max().item() < 0.02, "Policy must start at the cached reference."
print(f"Initial DPO loss: {initial_loss.mean().item():.4f}; expected {math.log(2):.4f}")


## Train and evaluate at fixed intervals

Three passes over 256 pairs, four pairs per microbatch, and two microbatches per optimizer update give 96 updates. Float16 gradient scaling prevents small gradients from underflowing; unscale before clipping. The reference scores remain fixed. Generation uses no cached preference answers.

After each epoch, score the same held-out requests. These measurements show the trajectory, but do not choose the training duration or alter the hyperparameters. To restart from scratch, rerun from model loading rather than only rerunning this cell.


In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
loader = DataLoader(list(range(len(train_pairs))), batch_size=BATCH_SIZE, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
assert len(loader) % ACCUMULATION_STEPS == 0
updates_per_epoch = len(loader) // ACCUMULATION_STEPS
total_updates = EPOCHS * updates_per_epoch
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.0)
scaler = torch.amp.GradScaler("cuda")
warmup_updates = max(1, total_updates // 10)
def lr_factor(step):
    if step < warmup_updates:
        return (step + 1) / warmup_updates
    return max(0.0, (total_updates - step) / (total_updates - warmup_updates))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
training_history = []
heldout_margins = [{"epoch": 0, "mean margin": 0.0, "positive margin (%)": 0.0}]
training_seconds = 0.0
successful_updates = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    model.config.use_cache = False
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    started = time.perf_counter()
    progress = tqdm(loader, desc=f"DPO epoch {epoch}/{EPOCHS}")
    for micro_step, indices in enumerate(progress):
        pairs = [train_pairs[i] for i in indices.tolist()]
        ref = reference_train[indices].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            chosen_logps, rejected_logps = pair_logps(model, pairs)
            losses, margin = dpo_loss(chosen_logps, rejected_logps, ref)
            loss = losses.mean()
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite loss; stop and inspect the run.")
        scaler.scale(loss / ACCUMULATION_STEPS).backward()
        if (micro_step + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                scheduler.step()
                successful_updates += 1
            optimizer.zero_grad(set_to_none=True)
        training_history.append({"epoch": epoch, "microbatch": len(training_history) + 1,
                                 "loss": loss.item(), "margin": margin.detach().mean().item()})
        progress.set_postfix(loss=f"{loss.item():.3f}")
    torch.cuda.synchronize()
    training_seconds += time.perf_counter() - started
    model.config.use_cache = True
    generated = generate_responses(model, test_records)
    outputs_by_epoch[epoch] = generated
    scored = evaluate(generated, test_records, f"DPO epoch {epoch}").assign(epoch=epoch)
    all_results.append(scored)
    current_test = cache_logps(model, test_pairs)
    _, test_margins = dpo_loss(current_test[:, 0], current_test[:, 1], reference_test)
    heldout_margins.append({"epoch": epoch, "mean margin": test_margins.mean().item(),
                           "positive margin (%)": 100 * (test_margins > 0).float().mean().item()})
    display(scored[metrics].mean().mul(100).round(1).to_frame("Held-out %"))

assert torch.equal(reference_train, reference_snapshot), "Reference cache changed."
print(f"Training time excluding evaluations: {training_seconds / 60:.2f} minutes.")
print(f"Completed optimizer updates: {successful_updates}/{total_updates}")


## Compare generated performance, not just training loss

The left chart shows whether DPO fits the preference pairs. The right chart measures what the model actually generates. Both matter: a falling pairwise loss can coexist with worse generations, particularly outside the training distribution.


In [ ]:
results = pd.concat(all_results, ignore_index=True)
summary = results.groupby("epoch", sort=True)[metrics].mean().mul(100)
history = pd.DataFrame(training_history)
display(summary.round(1).style.format("{:.1f}%"))
display(results.groupby(["epoch", "group"])[metrics].mean().mul(100).round(1))
display(pd.DataFrame(heldout_margins))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history["microbatch"], history["loss"], color="#b9a4ce", alpha=0.35)
axes[0].plot(history["microbatch"], history["loss"].rolling(8, min_periods=1).mean(), color="#8d6ca9")
axes[0].axhline(math.log(2), color="gray", linestyle=":", label="Initial loss: log 2")
axes[0].set(xlabel="Training microbatch", ylabel="DPO loss", title="Fitting preference pairs")
axes[0].legend()
for metric, color in zip(metrics, ["#b9a4ce", "#d5b780", "#83ab95"]):
    axes[1].plot(summary.index, summary[metric], "o-", label=metric, color=color)
axes[1].set(xlabel="Epoch (0 = before DPO)", ylabel="Held-out requests (%)", ylim=(-3, 103),
            xticks=list(range(EPOCHS + 1)), title="Generated-output performance")
axes[1].legend(fontsize=9)
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

final_scores = all_results[-1]
start_correct = before["Exact task success"].to_numpy(dtype=bool)
end_correct = final_scores["Exact task success"].to_numpy(dtype=bool)
delta = 100 * (end_correct.mean() - start_correct.mean())
print(f"Exact task success: {100 * start_correct.mean():.1f}% → {100 * end_correct.mean():.1f}% ({delta:+.1f} points).")
print("Fixed requests:", int((~start_correct & end_correct).sum()),
      "| Regressions:", int((start_correct & ~end_correct).sum()))
if delta <= 0:
    print("This run does not show a generation improvement. Preference margins alone do not establish one.")


## Inspect the answers

Show the first three requests from each held-out group, chosen independently of outcomes. Save every prediction, including failures, for comparison. If a response becomes shorter but loses correct fields, exact task success decreases rather than rewarding the shorter response.


In [ ]:
comparison = pd.DataFrame([
    {"group": r["group"], "request": r["request"], "expected": target_text(r),
     "before": outputs_by_epoch[0][i], "after": outputs_by_epoch[EPOCHS][i]}
    for i, r in enumerate(test_records)
])
selected = [0, 1, 2, TEST_SIZE // 2, TEST_SIZE // 2 + 1, TEST_SIZE // 2 + 2]
with pd.option_context("display.max_colwidth", None):
    display(comparison.iloc[selected].style.set_properties(**{
        "white-space": "pre-wrap", "text-align": "left", "vertical-align": "top",
    }))

output_dir = Path("dpo_order_results")
output_dir.mkdir(exist_ok=True)
comparison.to_csv(output_dir / "before_after.csv", index=False)
results.to_csv(output_dir / "all_epoch_predictions.csv", index=False)
history.to_csv(output_dir / "training_history.csv", index=False)
pd.DataFrame(heldout_margins).to_csv(output_dir / "heldout_margins.csv", index=False)
(output_dir / "run.json").write_text(json.dumps({
    "model": MODEL_ID, "revision": getattr(model.config, "_commit_hash", None),
    "seed": SEED, "beta": BETA, "learning_rate": LEARNING_RATE, "epochs": EPOCHS,
    "train_pairs": TRAIN_SIZE, "test_pairs": TEST_SIZE,
    "training_seconds": training_seconds, "successful_updates": successful_updates,
}, indent=2))
model.save_pretrained(output_dir / "adapter")
tokenizer.save_pretrained(output_dir / "adapter")
print("Saved adapter and results to", output_dir.resolve())


## What this experiment demonstrates

DPO learns from a fixed set of comparisons and a fixed reference. It does not need an explicit reward model, new rollouts during optimization, or a learned advantage. Teacher forcing lets us evaluate each recorded completion; the preference loss changes their relative probabilities.

A higher held-out exact-success rate shows improvement on this particular extraction-and-formatting task. Report both test groups and regressions, not just selected examples or the training loss. This small synthetic dataset does not establish broad helpfulness, human preference alignment, or superiority over SFT. In particular, comparing two unreplicated notebook runs is not a controlled algorithm benchmark.

### Sources

- [Direct Preference Optimization](https://arxiv.org/abs/2305.18290), especially the loss and Appendix B implementation.
- [Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B).
- [TRL DPO documentation](https://huggingface.co/docs/trl/dpo_trainer), including reference-log-probability precomputation. We write the small training loop explicitly to expose the mathematics.
